# 第 9 章 · SFT 精简总结

> 本文是 [ch09.ipynb](./ch09.ipynb) 的浓缩版。核心:**answer-only loss masking**。

## ⭐ loss masking 可视化

SFT 只从 **assistant 的回复 token** 学习,prompt(user/system)部分被标为 -100 忽略:

```
token:  <bos><|im_start|>user\n你好<|im_end|>\n<bos><|im_start|>assistant\n你好!<|im_end|>
label:  -100  -100 -100 -100 -100 -100 -100 -100 -100 你  好  !   <|im_end|>
        ──────────── 灰色(忽略) ──────────────── 绿色(参与 loss)──
```

`generate_labels` 算法:
1. 全部 label = -100
2. 找 `<bos>assistant\n` → `<eos>\n` 的 span
3. span 内 label = token_id

In [ ]:
import sys
sys.path.insert(0, '/home/minimind')
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('/home/minimind/model')

# 构造对话并渲染
messages = [
    {"role": "user", "content": "你好"},
    {"role": "assistant", "content": "你好!有什么可以帮助你的?"},
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
input_ids = tokenizer(prompt, add_special_tokens=False).input_ids

# generate_labels
bos_id = tokenizer(f'{tokenizer.bos_token}assistant\n', add_special_tokens=False).input_ids
eos_id = tokenizer(f'{tokenizer.eos_token}\n', add_special_tokens=False).input_ids

labels = [-100] * len(input_ids)
i = 0
while i < len(input_ids):
    if input_ids[i:i + len(bos_id)] == bos_id:
        start = i + len(bos_id)
        end = start
        while end < len(input_ids):
            if input_ids[end:end + len(eos_id)] == eos_id:
                break
            end += 1
        for j in range(start, min(end + len(eos_id), len(input_ids))):
            labels[j] = input_ids[j]
        i = end + len(eos_id)
    else:
        i += 1

# 可视化
GREEN, GRAY, RESET = '\033[92m', '\033[90m', '\033[0m'
print(f"{'idx':>3}  {'token':<16}  {'label':>8}  状态")
print("-" * 55)
for i, (tid, lab) in enumerate(zip(input_ids, labels)):
    tok = tokenizer.decode([tid]).replace('\n', '\\n')
    if lab != -100:
        print(f"{GREEN}{i:3d}  {tok:<16}  {lab:8d}  ✓ loss{RESET}")
    else:
        print(f"{GRAY}{i:3d}  {tok:<16}  {'-100':>8}  ✗ mask{RESET}")

## SFT vs Pretrain 对照表

| 维度 | 预训练 | SFT | 倍数 |
|---|---|---|---|
| **目标** | 学语言规律 | 学对话格式 | - |
| **学习率** | 5e-4 | **1e-5** | 低 50 倍 |
| **label 来源** | 全部 token | **仅 assistant token** | ~30-50% |
| **max_seq_len** | 340 | **768** | 2.3x |
| **from_weight** | none(从头) | **pretrain** | 继续训练 |
| **数据格式** | `{"text":"..."}` | `{"conversations":[...]}` | - |

> 核心:SFT 用低学习率在预训练基础上微调,只学 assistant 的回复模式。

## 下一步

SFT 让模型会对话 → 第 10 章 DPO 让模型回复**更好**。